# 🚀 AI Blog Generator - Colab (Self-Contained)
Generate professional blog posts using free GPU power!

**No git clone needed** — Everything runs in this notebook

## Step 1: Setup (Run Once)

In [ ]:
# Install required packages
!pip install -q langgraph langchain langchain-ollama langchain-core pydantic duckduckgo-search requests

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.ai/install.sh | sh

In [ ]:
# Start Ollama server in background
import subprocess
import time
import requests

# Start Ollama
proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("⏳ Starting Ollama server...")
time.sleep(5)

# Wait for server to be ready
for i in range(10):
    try:
        requests.get('http://localhost:11434/api/tags')
        print("✅ Ollama server is ready")
        break
    except:
        time.sleep(1)
        if i == 9:
            print("⚠️ Ollama taking longer than expected...")

In [ ]:
# Download Mistral model (7B, fast, good quality)
print("📥 Downloading Mistral model (~4GB)...")
print("This takes 2-3 minutes (one-time download)\n")

!ollama pull mistral

print("\n✅ Model ready!")

## Step 2: Define Models & Functions

In [ ]:
# Define data models
from pydantic import BaseModel, Field
from typing import Optional

class GraphState(BaseModel):
    """State passed through the pipeline"""
    topic: str
    mode: str = "open_book"  # open_book, hybrid, closed_book
    research_data: Optional[list] = None
    plan: Optional[dict] = None
    sections: Optional[list] = None
    blog: Optional[str] = None
    errors: list = Field(default_factory=list)
    
    class Config:
        arbitrary_types_allowed = True

print("✅ Models defined")

In [ ]:
# Setup LLM
from langchain_ollama import OllamaLLM

def get_llm(temperature: float = 0.7):
    """Return an OllamaLLM instance"""
    return OllamaLLM(
        model="mistral",
        base_url="http://localhost:11434",
        temperature=temperature,
        num_predict=4096,
    )

# Test it
print("Testing LLM...")
llm = get_llm()
response = llm.invoke("Say 'Blog generator ready!' in 2 words")
print(f"LLM Response: {response}")
print("✅ LLM working!")

In [ ]:
# Search function using DuckDuckGo
from duckduckgo_search import DDGS
import json
import logging

logger = logging.getLogger("blog_gen")

def search_topic(query: str, max_results: int = 5) -> list[dict]:
    """Search using DuckDuckGo (free, no API key)"""
    try:
        ddgs = DDGS()
        results = ddgs.text(query, max_results=max_results)
        return [
            {
                "title": r.get("title", ""),
                "url": r.get("href", ""),
                "content": r.get("body", "")[:500],  # Truncate
            }
            for r in results
        ]
    except Exception as e:
        logger.error(f"Search failed: {e}")
        return []

# Test search
print("Testing search...")
results = search_topic("artificial intelligence", max_results=2)
print(f"Found {len(results)} results")
print("✅ Search working!")

In [ ]:
# Utility functions
import re

def extract_json(text: str) -> dict | list:
    """Extract JSON from LLM response"""
    try:
        # Try direct JSON parse
        return json.loads(text)
    except:
        # Try to find JSON in the text
        match = re.search(r'\[.*\]|\{.*\}', text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except:
                return {}
        return {}

def clean_markdown(text: str) -> str:
    """Clean up markdown formatting"""
    # Remove extra backticks and code blocks
    text = re.sub(r'^```.*?\n', '', text, flags=re.MULTILINE)
    text = re.sub(r'\n```$', '', text)
    return text.strip()

print("✅ Utilities loaded")

## Step 3: Define Pipeline Nodes

In [ ]:
# Router node - decides on mode
def router_node(state: GraphState) -> GraphState:
    """Route based on mode"""
    mode = state.mode
    topic = state.topic
    print(f"🔀 Router: mode={mode}, topic='{topic}'")
    return state

# Research node - gathers info
def research_node(state: GraphState) -> GraphState:
    """Research the topic"""
    if state.mode == "closed_book":
        return state
    
    print(f"🔍 Researching: '{state.topic}'...")
    results = search_topic(state.topic, max_results=5)
    
    # Summarize with LLM
    if results:
        sources = "\n".join(
            f"- {r['title']}: {r['content']}"
            for r in results[:3]
        )
        
        llm = get_llm(temperature=0.2)
        prompt = f"""Summarize these sources about '{state.topic}' in 2-3 key points.
        
Sources:
{sources}
,
""
        
        summary = llm.invoke(prompt)
        state.research_data = [{
            "title": "Research Summary",
            "content": summary
        }]
    
    print(f"✅ Research complete: {len(state.research_data or [])} items")
    return state

print("✅ Router & Research nodes defined")

In [ ]:
# Orchestrator node - creates plan
def orchestrator_node(state: GraphState) -> GraphState:
    """Create blog outline/plan"""
    print(f"📋 Creating plan for '{state.topic}'...")
    
    llm = get_llm(temperature=0.5)
    
    research_context = ""
    if state.research_data:
        research_context = f"\n\nResearch context:\n{state.research_data[0].get('content', '')}"
    
    prompt = f"""Create a blog post outline for '{state.topic}'.
{research_context}
,
  "intro": "Brief introduction (2 sentences)",
  "sections": [
    {{"heading": "Section 1", "points": ["point1", "point2"]}},
    {{"heading": "Section 2", "points": ["point1", "point2"]}},
    {{"heading": "Section 3", "points": ["point1", "point2"]}}
  ],
  "conclusion": "Brief conclusion"
}}"""
    
    response = llm.invoke(prompt)
    plan = extract_json(response)
    
    if not isinstance(plan, dict):
        plan = {"title": state.topic, "sections": []}
    
    state.plan = plan
    print(f"✅ Plan created: {plan.get('title', 'Untitled')}")
    return state

print("✅ Orchestrator node defined")

In [ ]:
# Writer node - writes sections
def writer_node(state: GraphState) -> GraphState:
    """Write blog sections"""
    if not state.plan:
        return state
    
    print(f"✍️ Writing sections...")
    
    llm = get_llm(temperature=0.7)
    sections = []
    
    for sec in state.plan.get("sections", [])[:3]:  # Max 3 sections
        heading = sec.get("heading", "")
        points = sec.get("points", [])
        
        prompt = f"""Write a detailed blog section:
Title: {heading}
Topic: {state.topic}
Key points to cover: {', '.join(points)}

Write 3-4 paragraphs with concrete examples. Keep it engaging and informative."""
        
        content = llm.invoke(prompt)
        sections.append({
            "heading": heading,
            "content": content
        })
        print(f"  ✅ {heading}")
    
    state.sections = sections
    print(f"✅ Wrote {len(sections)} sections")
    return state

print("✅ Writer node defined")

In [ ]:
# Reducer/Assembler node - combines into final blog
def assembler_node(state: GraphState) -> GraphState:
    """Assemble final blog post"""
    print(f"🔗 Assembling blog...")
    
    parts = []
    
    # Title
    title = state.plan.get("title", state.topic) if state.plan else state.topic
    parts.append(f"# {title}\n")
    
    # Intro
    if state.plan:
        intro = state.plan.get("intro", "")
        if intro:
            parts.append(f"\n{intro}\n")
    
    # Sections
    if state.sections:
        for sec in state.sections:
            parts.append(f"\n## {sec['heading']}\n")
            parts.append(f"\n{sec['content']}\n")
    
    # Conclusion
    if state.plan:
        conclusion = state.plan.get("conclusion", "")
        if conclusion:
            parts.append(f"\n## Conclusion\n")
            parts.append(f"\n{conclusion}\n")
    
    blog = "".join(parts)
    state.blog = clean_markdown(blog)
    
    print(f"✅ Blog assembled: {len(state.blog)} characters")
    return state

print("✅ Assembler node defined")

## Step 4: Generate Your Blog

In [ ]:
# Configure your blog
BLOG_TOPIC = "Artificial Intelligence in Healthcare"  # ← CHANGE THIS
BLOG_MODE = "open_book"  # open_book (research + LLM), closed_book (LLM only)

print(f"📝 Generating blog: '{BLOG_TOPIC}'")
print(f"📚 Mode: {BLOG_MODE}")
print(f"\nThis will take 3-5 minutes...\n")

In [ ]:
# Run the pipeline
state = GraphState(
    topic=BLOG_TOPIC,
    mode=BLOG_MODE,
    errors=[]
)

print("="*60)
print("PIPELINE EXECUTION")
print("="*60)

# Run nodes in sequence
state = router_node(state)
state = research_node(state)
state = orchestrator_node(state)
state = writer_node(state)
state = assembler_node(state)

print("\n" + "="*60)
print("✅ PIPELINE COMPLETE")
print("="*60)

In [ ]:
# Display the blog
print("\n" + "="*80)
print("📖 YOUR GENERATED BLOG")
print("="*80 + "\n")
print(state.blog)

In [ ]:
# Save the blog
filename = "generated_blog.md"

with open(filename, "w") as f:
    f.write(state.blog)

print(f"✅ Saved to: {filename}")
print(f"📊 Size: {len(state.blog)} characters")
print(f"📄 Sections: {len(state.sections or [])}")

In [ ]:
# Download the blog
from google.colab import files

print("\n💾 Downloading to your computer...")
files.download('generated_blog.md')
print("✅ Downloaded!")

## (Optional) Generate Multiple Blogs

In [ ]:
# Generate multiple blogs
topics = [
    "Quantum Computing",
    "Sustainable Energy Solutions",
    "Mental Health in the Digital Age"
]

for i, topic in enumerate(topics, 1):
    print(f"\n{'='*60}")
    print(f"[{i}/{len(topics)}] {topic}")
    print('='*60)
    
    state = GraphState(topic=topic, mode="open_book", errors=[])
    
    state = router_node(state)
    state = research_node(state)
    state = orchestrator_node(state)
    state = writer_node(state)
    state = assembler_node(state)
    
    # Save
    filename = f"blog_{i}_{topic.replace(' ', '_')}.md"
    with open(filename, "w") as f:
        f.write(state.blog)
    print(f"✅ Saved: {filename}")

print(f"\n✅ Generated {len(topics)} blogs!")